# Stochastic PDE Datasets (Preview)

> **Note**: This notebook previews upcoming features. Stochastic models are under development.

This notebook outlines how PDEForge will support **stochastic PDEs** for uncertainty quantification and generative modeling.

## Why Stochastic PDEs?

Real physical systems have **inherent randomness**:

- Thermal fluctuations
- Turbulent forcing
- Measurement noise
- Uncertain parameters

Stochastic PDEs model this:

$$\frac{\partial u}{\partial t} = \mathcal{L}[u] + \sigma \dot{W}$$

where $\dot{W}$ is a noise process.

**The same input can produce different outputs!**

## The Challenge for Operator Learning

| Deterministic PDE | Stochastic PDE |
|-------------------|----------------|
| $f(\text{input}) = \text{output}$ | $f(\text{input}, \omega) = \text{output}(\omega)$ |
| Unique solution | Distribution of solutions |
| Learn the mapping | Learn the conditional distribution |

What should the neural operator learn?

## Two Output Formats

PDEForge will support two formats for stochastic outputs:

### Format 1: Multiple Realizations

For **generative models** (VAEs, diffusion models, score matching):

```python
# Coming soon!
dataset = generate_dataset(
    model="stochastic_heat_2d",
    n_samples=1000,
    resolution={"x": 64, "y": 64},
    params={
        "diffusivity": 0.1,
        "noise_intensity": 0.05,
        "n_realizations": 50,  # 50 noise realizations per IC
    },
)

# Shapes:
# dataset.inputs.shape = (1000, 64, 64)           # Initial conditions
# dataset.outputs.shape = (1000, 50, 64, 64)      # 50 realizations each
```

**Learning task**: Learn $p(\text{output} | \text{input})$

### Format 2: Moments

For **UQ models** (mean + variance prediction):

```python
# Coming soon!
dataset = generate_dataset(
    model="stochastic_heat_2d",
    n_samples=1000,
    resolution={"x": 64, "y": 64},
    params={
        "diffusivity": 0.1,
        "noise_intensity": 0.05,
        "output_moments": True,
        "n_realizations": 100,  # More for stable statistics
    },
)

# Shapes:
# dataset.inputs.shape = (1000, 64, 64)           # Initial conditions  
# dataset.output_mean.shape = (1000, 64, 64)      # E[output | input]
# dataset.output_var.shape = (1000, 64, 64)       # Var[output | input]
```

**Learning task**: Learn $\mathbb{E}[\text{output} | \text{input}]$ and $\text{Var}[\text{output} | \text{input}]$

## Planned Stochastic Models

### 1. Stochastic Heat Equation

$$\frac{\partial u}{\partial t} = \alpha \nabla^2 u + \sigma \dot{W}$$

- **Additive noise**: Noise added to the dynamics
- **Linear**: Well-understood analytical properties
- **Good test case**: Verify moment estimation

### 2. Stochastic Burgers Equation

$$\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2} + \sigma \dot{W}$$

- **Nonlinear**: Shocks interact with noise
- **Multiplicative option**: $\sigma(u) \dot{W}$
- **Rich dynamics**: Test generative models

### 3. Stochastic Allen-Cahn

$$\frac{\partial u}{\partial t} = \epsilon \nabla^2 u + u - u^3 + \sigma \dot{W}$$

- **Bistable**: Noise can trigger phase transitions
- **Pattern formation**: Interesting spatial structure

## Noise Parameters

Stochastic models will expose noise-related parameters:

| Parameter | Description | Effect |
|-----------|-------------|--------|
| `noise_intensity` | Strength of noise (σ) | Higher → more variance |
| `noise_correlation_length` | Spatial smoothness | 0 = white, >0 = smooth |
| `noise_type` | "additive" or "multiplicative" | How noise couples to state |

## Use Case: Training a UQ-Aware Neural Operator

```python
# Coming soon - conceptual workflow

# 1. Generate stochastic dataset
dataset = generate_dataset(
    model="stochastic_burgers_1d",
    n_samples=5000,
    params={"output_moments": True},
)

splits = dataset.split(train=0.6, val=0.15, cal=0.15, test=0.1)

# 2. Train model to predict mean AND variance
from operator_uq import FNO_UQ

model = FNO_UQ(output_mean=True, output_var=True)
model.fit(
    X=splits['train'].inputs,
    y_mean=splits['train'].output_mean,
    y_var=splits['train'].output_var,
)

# 3. Predictions come with learned uncertainty
pred_mean, pred_var = model.predict(test_inputs)

# pred_var reflects the INHERENT stochasticity of the PDE,
# not just model uncertainty!
```

## Use Case: Generative Modeling

```python
# Coming soon - conceptual workflow

# 1. Generate with multiple realizations
dataset = generate_dataset(
    model="stochastic_heat_2d",
    n_samples=2000,
    params={"n_realizations": 50},
)

# 2. Train conditional generative model
from your_generative_lib import ConditionalDiffusion

gen_model = ConditionalDiffusion()
gen_model.fit(
    conditions=dataset.inputs,      # (2000, nx, ny)
    samples=dataset.outputs,        # (2000, 50, nx, ny)
)

# 3. Sample new realizations given a new IC
new_ic = ...  # Your new initial condition
samples = gen_model.sample(condition=new_ic, n_samples=100)

# samples.shape = (100, nx, ny) - 100 possible outcomes!
```

## Visualization (Planned)

The interactive explorer will support stochastic data:

```python
from pdeforge.visualization import StochasticExplorer

explorer = StochasticExplorer(dataset)
explorer.show()

# Features:
# - Slider to browse realizations
# - Toggle between realizations / mean / std views
# - Overlay multiple realizations
```

## Current Status & Roadmap

| Feature | Status |
|---------|--------|
| `StochasticPDEModel` base class | Planned |
| `stochastic_heat_2d` | Planned |
| `stochastic_burgers_1d` | Planned |
| Moment-based output format | Planned |
| Realization-based output format | Planned |
| Stochastic data explorer | Planned |

**Coming in a future release!**

## For Now: Deterministic Models

While stochastic models are in development, you can:

1. Use deterministic models with the UQ workflow (see `03_uq_workflow.ipynb`)
2. Apply MC Dropout or ensemble methods for epistemic uncertainty
3. Use conformal prediction for coverage guarantees

These capture **model uncertainty**, while stochastic PDEs will capture **aleatoric uncertainty** (inherent randomness).